In [17]:
!pip install --upgrade gradio langchain===1.2.13 langchain-core===1.2.23 langchain_groq===1.1.2 google-search-results===2.4.2 langchain-community===0.4.1 langchain-text-splitters===1.1.1 langchain-huggingface===1.2.1 pypdf faiss-cpu langchain_huggingface

Load the File and split it

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
loader = PyPDFLoader("nestle_hr_policy.pdf")
documents = loader.load_and_split(RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64))

In [8]:
# print(documents[0])
print(f"length of document {len(documents)} - content[0] {documents[0].page_content}")

length of document 35 - content[0] Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy


Generate Embeddings

In [9]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
embeddings = hf.embed_documents([doc.page_content for doc in documents])
print(f"embedding  len {len(embeddings)} - sample data - {embeddings[0]}")

embedding  len 35 - sample data - [0.051310136914253235, 0.029005806893110275, 0.013930663466453552, -0.06950163096189499, 0.03482823073863983, 0.029292652383446693, 0.021085189655423164, 0.009378933347761631, 0.09868790954351425, -0.02176043763756752, 0.1223379522562027, -0.0056806583888828754, -0.026832478120923042, 0.012239724397659302, 0.03352859243750572, 0.002343796193599701, 0.006689908914268017, 0.01872226595878601, 0.058524277061223984, 0.020070334896445274, -0.04935595393180847, 0.013227547518908978, 0.031912438571453094, 0.008116187527775764, -0.0017091181362047791, -0.003738157916814089, 0.04506663978099823, -0.006825411692261696, 0.016872359439730644, -0.05422639474272728, 0.07013417035341263, 0.015876207500696182, -0.02641497179865837, -0.09233970195055008, 1.7247771211259533e-06, -0.022893382236361504, 0.038953058421611786, 0.035027049481868744, -0.013991639018058777, -0.005954289343208075, 0.025256117805838585, -0.03620268777012825, -0.020736178383231163, 0.053963214159

In [13]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(documents, hf)
retriever=vectorstore.as_retriever()

In [14]:
query2 = "what are line managers responibility?"
docs = retriever.invoke(query2)
for doc in docs:
    print(doc.page_content)

The Nestlé Human Resources Policy
2
Line managers have the prime responsibility for 
building and sustaining an environment where 
people have a sense of personal commitment 
to their work and give their best to ensure our 
Company’s success. They care for and develop 
the leaders of tomorrow.
Line managers decide on all people matters 
under their influence, within the boundaries set 
by the policies and principles, acting as the final 
decision makers.
The Human Resources (HR) structure
lenging responsibilities and ensuring that employ-
ees are aware of how their work impacts Nestlé.
The line manager and employee work together 
to ensure that challenging objectives are set and  
effectively evaluated throughout the year. This 
further enables managers to acknowledge high 
performance and reward employees accordingly, 
while ensuring low performance is properly 
managed with integrity.
Employees receive regular feedback on their 
performance and career aspirations through
its own empl

In [28]:
from langchain_groq import ChatGroq
from google.colab import userdata
import os
from langchain_core.prompts import PromptTemplate
def get_prompt_template():
  prompt_template = PromptTemplate.from_template(
        """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use one sentence and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
    )
  return prompt_template

def get_llm(stream_response=False):
  os.environ["GROQ_API_KEY"] =userdata.get('LLM_API_KEY')
  return ChatGroq(
      model="openai/gpt-oss-120b",
      temperature=0,
      max_completion_tokens=8192,
      top_p=1,
      reasoning_effort="medium",
      stream=stream_response,
      stop=None
  )

In [34]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
prompt_template = get_prompt_template()
# prompt = prompt_template.format(**prompt_input)
llm = get_llm()
output_parser=StrOutputParser()

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | output_parser
)

def process_user_query(user_query):
  response = rag_chain.invoke(user_query)
  return response

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! max_completion_tokens is not default parameter.
                    max_completion_tokens was transferred to model_kwargs.
                    Please confirm that max_completion_tokens is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: UserWarning: WARNING! stream is not default parameter.
                    stream was transferred to model_kwargs.
                    Please confirm that stream is what you intended.
  validated_self = self.__pydantic_validat

In [32]:
rag_chain.invoke("what are line managers responibility?")

'Line managers are responsible for building a committed work environment, developing future leaders, making final decisions on people matters, setting and evaluating objectives, and managing performance with integrity.'

In [40]:
process_user_query("what are line managers responibility?")

'Line managers are responsible for building a committed work environment, developing future leaders, making final decisions on people matters, setting and evaluating objectives, and managing performance with integrity.'

In [51]:
import html
import gradio as gr


def answer_fn(user_input, history):
    return process_user_query(user_input)


def render_chat(history):
    parts = ['<div class="chat-scroll">']

    for user_msg, bot_msg in history:
        if user_msg:
            parts.append(
                f"""
                <div class="msg-row user-row">
                    <div class="msg user-msg">{html.escape(user_msg)}</div>
                </div>
                """
            )

        if bot_msg:
            bot_html = html.escape(bot_msg).replace("\n", "<br>")
            parts.append(
                f"""
                <div class="msg-row bot-row">
                    <div class="msg bot-msg">{bot_html}</div>
                </div>
                """
            )

    parts.append("</div>")
    return "".join(parts)


def chat(user_message, history):
    history = history or []
    user_message = (user_message or "").strip()

    if not user_message:
        return render_chat(history), history, ""

    response = answer_fn(user_message, history)
    history.append((user_message, response))

    return render_chat(history), history, ""


with gr.Blocks(fill_height=True) as demo:
    gr.HTML("<div class='app-title'>🤖 AI Powered HR Assistant 🤖 </div>")

    history_state = gr.State([])

    chat_html = gr.HTML(
        value=render_chat([]),
        elem_id="chat-panel"
    )

    with gr.Row(elem_id="composer-row"):
        msg = gr.Textbox(
            placeholder="Ask anything...",
            show_label=False,
            lines=1,
            max_lines=6,
            container=False,
            scale=30,
            elem_id="composer-input"
        )
        send = gr.Button(
            "➜",
            elem_id="send-btn",
            min_width=58
        )

    msg.submit(chat, [msg, history_state], [chat_html, history_state, msg])
    send.click(chat, [msg, history_state], [chat_html, history_state, msg])

demo.launch(
    css="""
    :root {
        --bg: #081225;
        --panel: #0f1728;
        --panel-soft: #0b1a36;
        --border: #22314d;
        --text: #e5e7eb;
        --muted: #94a3b8;
        --user: #2f6df6;
        --assistant: #13233d;
        --send: #f97316;
    }

    html, body {
        margin: 0;
        padding: 0;
        height: 100%;
        background: var(--bg);
        overflow: hidden;
    }

    body, .gradio-container {
        background: var(--bg) !important;
        color: var(--text) !important;
        height: 100vh !important;
        overflow: hidden !important;
    }

    .gradio-container {
        max-width: 100% !important;
        padding: 16px 18px 18px 18px !important;
        box-sizing: border-box !important;
    }

    .app-title {
        width: 100%;
        margin: 0 0 12px 0;
        font-size: 22px;
        font-weight: 700;
        color: var(--text);
    }

    #chat-panel {
        width: 100%;
        height: calc(90vh - 130px);
        border: 1px solid var(--border);
        border-radius: 20px;
        background: linear-gradient(180deg, #101722 0%, #0a1426 100%);
        overflow: hidden;
        box-sizing: border-box;
    }

    .chat-scroll {
        height: 100%;
        overflow-y: auto;
        padding: 18px;
        box-sizing: border-box;
    }

    .msg-row {
        display: flex;
        width: 100%;
        margin-bottom: 14px;
    }

    .user-row {
        justify-content: flex-end;
    }

    .bot-row {
        justify-content: flex-start;
    }

    .msg {
        padding: 14px 18px;
        border-radius: 18px;
        line-height: 1.55;
        font-size: 16px;
        border: 1px solid var(--border);
        box-sizing: border-box;
        word-break: break-word;
        overflow-wrap: anywhere;
        white-space: pre-wrap;
    }

    .user-msg {
        background: var(--user);
        color: white;
        border-color: transparent;
        max-width: 28%;
        min-width: fit-content;
    }

    .bot-msg {
        background: var(--assistant);
        color: var(--text);
        max-width: 74%;
    }

    #composer-row {
        width: 100%;
        margin: 10px 0 0 0 !important;
        gap: 10px;
        align-items: center;
        display: flex !important;
        flex-wrap: nowrap !important;
    }

    #composer-input {
        flex: 1 1 auto !important;
        min-width: 0 !important;
    }

    #composer-input textarea {
        background: var(--panel-soft) !important;
        color: var(--text) !important;
        border: 1px solid var(--border) !important;
        border-radius: 16px !important;
        padding: 14px 16px !important;
        min-height: 56px !important;
        font-size: 16px !important;
        line-height: 1.4 !important;
        box-shadow: none !important;
    }

    #composer-input textarea::placeholder {
        color: var(--muted) !important;
    }

    #send-btn {
        height: 56px !important;
        min-width: 58px !important;
        max-width: 58px !important;
        border-radius: 14px !important;
        font-size: 22px !important;
        background: var(--send) !important;
        border: none !important;
        flex: 0 0 58px !important;
    }

    footer {
        display: none !important;
    }

    @media (max-width: 768px) {
        .gradio-container {
            padding: 12px !important;
        }

        #chat-panel {
            height: calc(100vh - 122px);
        }

        .user-msg {
            max-width: 50%;
        }

        .bot-msg {
            max-width: 92%;
        }

        #send-btn {
            min-width: 54px !important;
            max-width: 54px !important;
            height: 54px !important;
            flex-basis: 54px !important;
        }
    }
    """
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70ac8fe4afceda544e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [35]:
import gradio as gr

# -------------------------------
# Your backend logic (replace this)
# -------------------------------
def answer_fn(user_input, history):
    # Example: replace with your RAG / API / LLM call
    response = user_query(user_input)
    return response


# -------------------------------
# Chat handler
# -------------------------------
def chat(user_message, history):
    history = history or []

    response = answer_fn(user_message, history)

    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": response})

    return history, ""


# -------------------------------
# UI
# -------------------------------
with gr.Blocks(
    theme=gr.themes.Soft(),
    css="""
    .gradio-container {
        max-width: 800px !important;
        margin: auto;
    }
    """
) as demo:

    gr.Markdown("## 🤖 Chat Assistant")

    chatbot = gr.Chatbot(
        type="messages",
        height=500,
        show_label=False
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask something...",
            show_label=False,
            scale=4
        )
        send = gr.Button("Send", variant="primary", scale=1)

    state = gr.State([])

    # Submit via Enter
    msg.submit(chat, [msg, state], [chatbot, msg]).then(
        lambda h: h, chatbot, state
    )

    # Submit via Button
    send.click(chat, [msg, state], [chatbot, msg]).then(
        lambda h: h, chatbot, state
    )

# -------------------------------
# Run
# -------------------------------
demo.launch(debug=True)

/tmp/ipykernel_1131/2177146548.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


TypeError: Chatbot.__init__() got an unexpected keyword argument 'type'